# Pandas — Phase 0: The Architectural Foundations
### Credit Card Risk Analysis Track

You jumped straight into loading and manipulating data in Phases 1-6 — which worked, because pandas is intuitive enough to use before you fully understand it. This phase fills in the conceptual floor underneath all of that: what a Series and DataFrame actually *are*, why some things are `df.shape` and others are `df.head()`, why loops are the wrong tool, and what the Index is really doing every time you filter or merge.

**Topics in this phase:**
- 0A. The Pandas Series (1D Arrays)
- 0B. The Pandas DataFrame (2D Tabular Grids)
- 0C. DataFrame Methods vs. Attributes
- 0D. Vectorization Principles (why we don't use for loops)
- 0E. The Index Object — `.set_index()`, `.reset_index()`

**Dataset:** the clean `loan_applications.csv` from Phase 1. Keep it in the same folder as this notebook.

**How to use this notebook:**
- Each question has a `YOUR CODE HERE` cell — attempt it first.
- The `Solution` cell right after shows one correct approach — compare, don't just copy.
- All solutions were run against the actual dataset before this notebook was assembled.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import time
print(pd.__version__)

## Topic 0A: The Pandas Series (1D Arrays)

A **Series** is a single labeled column: a 1D array of values (backed by numpy) plus an **index** of labels running alongside it. Every column you've pulled out of a DataFrame in earlier phases has secretly been a Series.

**Q1.** Create a Series `loan_amounts` from the list `[12000, 8500, 21000, 5000, 15500]`, with a custom index `["A101", "A102", "A103", "A104", "A105"]` representing applicant IDs. Print it.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
loan_amounts = pd.Series(
    [12000, 8500, 21000, 5000, 15500], index=["A101", "A102", "A103", "A104", "A105"]
)
print(loan_amounts)

**Q2.** Create a Series `scores_series` directly from the dict `score_dict = {"A101": 720, "A102": 640, "A103": 800}`. Notice the dict's keys automatically become the index.

In [ ]:
score_dict = {"A101": 720, "A102": 640, "A103": 800}

# YOUR CODE HERE


**Solution**

In [ ]:
scores_series = pd.Series(score_dict)
print(scores_series)

**Q3.** From `loan_amounts` (Q1), get the value for applicant `"A103"` two ways: by **label** (`loan_amounts["A103"]`) into `by_label`, and by **position** (`loan_amounts.iloc[2]`) into `by_position`. Confirm they're equal — same underlying data, two different ways to address it, exactly like `.loc` vs `.iloc` on a DataFrame.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
by_label = loan_amounts["A103"]
by_position = loan_amounts.iloc[2]
print(by_label, by_position, by_label == by_position)

**Q4.** A Series is really two parts glued together. Extract just the raw data as `values_array` using `.values` (this is a plain numpy array — everything from the numpy phase applies to it), and just the labels as `index_obj` using `.index`. Print both types/contents.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
values_array = loan_amounts.values
index_obj = loan_amounts.index
print(type(values_array), list(index_obj))

**Q5.** This is the single most important thing that separates a Series from a plain numpy array: arithmetic **aligns by label**, not position. Given `series_a` (index `A101,A102,A103`) and `series_b` (index `A102,A103,A104`) below, compute `aligned_sum = series_a + series_b`. Look closely at which labels get real sums and which get `NaN` — pandas matched up `A102` and `A103` (present in both) and produced `NaN` for `A101` and `A104` (present in only one).

In [ ]:
series_a = pd.Series([100, 200, 300], index=["A101", "A102", "A103"])
series_b = pd.Series([10, 20, 30], index=["A102", "A103", "A104"])

# YOUR CODE HERE


**Solution**

In [ ]:
aligned_sum = series_a + series_b
print(aligned_sum)

**Q6.** Load `loan_applications.csv` into `df_temp`, then pull out the `credit_score` column into `credit_score_series`. Confirm it's actually a `pd.Series` (using `isinstance`... or just `type()`), and confirm its index is identical to `df_temp`'s index using `.index.equals()` — a DataFrame column always shares the parent DataFrame's index.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df_temp = pd.read_csv("loan_applications.csv")
credit_score_series = df_temp["credit_score"]
print(type(credit_score_series), credit_score_series.index.equals(df_temp.index))

## Topic 0B: The Pandas DataFrame (2D Tabular Grids)

A **DataFrame** is best understood as a dict of Series that all share the same index — each column is a Series, and the DataFrame just keeps them aligned side by side.

**Q7.** Build `df_from_dict` from a dict where each key is a column name and each value is a list of that column's values (`applicant`, `loan_amount`, `credit_score` for 3 rows). This is the most common way to construct a small DataFrame by hand.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df_from_dict = pd.DataFrame({
    "applicant": ["A101", "A102", "A103"],
    "loan_amount": [12000, 8500, 21000],
    "credit_score": [720, 640, 800],
})
print(df_from_dict)

**Q8.** Given `records`, a **list of dicts** (one dict per row — this is what `pd.read_json` or an API response often looks like), build `df_from_records` with `pd.DataFrame(records)`.

In [ ]:
records = [
    {"applicant": "A101", "loan_amount": 12000},
    {"applicant": "A102", "loan_amount": 8500},
]

# YOUR CODE HERE


**Solution**

In [ ]:
df_from_records = pd.DataFrame(records)
print(df_from_records)

**Q9.** Prove the "dict of aligned Series" mental model from the intro: confirm `df_from_dict["loan_amount"]` is a `pd.Series` (`col_is_series`), and confirm that column's index is identical to `df_from_dict`'s own index (`same_index`).

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
col_is_series = isinstance(df_from_dict["loan_amount"], pd.Series)
same_index = df_from_dict["loan_amount"].index.equals(df_from_dict.index)
print(col_is_series, same_index)

**Q10.** Build `df_custom_index`: a DataFrame with one column `loan_amount` (`[12000, 8500, 21000]`), but pass `index=["A101", "A102", "A103"]` at construction time instead of using the default `0, 1, 2` row numbers.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df_custom_index = pd.DataFrame(
    {"loan_amount": [12000, 8500, 21000]}, index=["A101", "A102", "A103"]
)
print(df_custom_index)

**Q11.** Given two separate Series, `amounts` and `scores` (both with the default `0,1,2` index), combine them into one DataFrame `df_from_series` by passing `{"loan_amount": amounts, "credit_score": scores}` to `pd.DataFrame(...)` — this is the "dict of Series" construction made literal.

In [ ]:
amounts = pd.Series([12000, 8500, 21000], name="loan_amount")
scores = pd.Series([720, 640, 800], name="credit_score")

# YOUR CODE HERE


**Solution**

In [ ]:
df_from_series = pd.DataFrame({"loan_amount": amounts, "credit_score": scores})
print(df_from_series)

## Topic 0C: DataFrame Methods vs. Attributes

You've been typing `df.shape` (no parentheses) and `df.head()` (parentheses) since Phase 1 without necessarily stopping to ask why. An **attribute** is a stored property, computed once, accessed directly. A **method** is a function bound to the object — it needs `()` to actually run, and often takes arguments.

Run this setup cell first — the rest of the notebook uses this `df`:

In [ ]:
df = pd.read_csv("loan_applications.csv", parse_dates=["application_date"])
print(df.shape)

**Q12.** Get `df.shape` into `shape_attr` (no parentheses — this is an attribute). Then, inside a `try`/`except TypeError`, attempt to *call* it as `df.shape()` — this should raise a `TypeError`, because a tuple (which is what `.shape` actually is) isn't callable. Print the attribute value and confirm the error type.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
shape_attr = df.shape
try:
    bad_call = df.shape()
    print("no error (unexpected)")
except TypeError as e:
    print(shape_attr, "TypeError raised as expected:", type(e).__name__)

**Q13.** Collect five common **attributes** into a dict `attrs`: `columns`, `index`, `dtypes`, `ndim` (number of dimensions — 2 for a DataFrame), and `size` (total cell count — rows × columns). None of these need parentheses. Print it.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
attrs = {
    "columns": df.columns,
    "index": df.index,
    "dtypes": df.dtypes,
    "ndim": df.ndim,
    "size": df.size,
}
print(attrs["ndim"], attrs["size"], list(attrs["columns"])[:3])

**Q14.** Now call three **methods** that do need parentheses: `df.head(2)` into `head_result`, `df["loan_amount"].sum()` into `sum_result`, and `df.sort_values("loan_amount")` into `sorted_result`. Print their types/shapes to see they each *do* something (compute, transform, filter) rather than just reporting a stored fact.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
head_result = df.head(2)
sum_result = df["loan_amount"].sum()
sorted_result = df.sort_values("loan_amount")
print(type(head_result), type(sum_result), sorted_result.shape)

**Q15.** Some attributes are directly **settable** — you can assign straight to them. On a copy `df_renamed_copy`, reassign `.columns` to an uppercased version of the existing column names (a list comprehension). Print a few of the new column names alongside the original `df`'s unchanged columns, confirming your copy didn't affect the original.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df_renamed_copy = df.copy()
df_renamed_copy.columns = [c.upper() for c in df_renamed_copy.columns]
print(df_renamed_copy.columns[:3].tolist(), df.columns[:3].tolist())

**Q16.** Most methods **return a new object** by default and leave the original untouched — that's why Phase 4 always did `df = df.rename(...)` rather than just `df.rename(...)`. On a copy `df_sort_practice`, call `.sort_values("loan_amount")` **without** reassigning it, into `result_returned` — confirm `df_sort_practice` itself is unchanged (still original order) by comparing first-row indices. Then call `.sort_values("loan_amount", inplace=True)` on `df_sort_practice` directly (no assignment) — now confirm it *does* match `result_returned`, because `inplace=True` mutates the object itself instead of returning a new one.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df_sort_practice = df.copy()
result_returned = df_sort_practice.sort_values("loan_amount")
print("no inplace:", df_sort_practice.index[0] == result_returned.index[0])

df_sort_practice.sort_values("loan_amount", inplace=True)
print("with inplace:", df_sort_practice.index[0] == result_returned.index[0])

## Topic 0D: Vectorization Principles

You saw this exact idea in the numpy mini-project — it applies just as much to pandas, since a Series is a numpy array underneath. Every `for` loop over rows you're tempted to write in pandas almost certainly has a faster vectorized equivalent.

**Q17.** To make the timing difference obvious rather than noisy on a small dataset, build `big_amounts`: `loan_amount` values tiled 2000× into a 1,000,000-row Series (`pd.Series(np.tile(df["loan_amount"].fillna(0).values, 2000))`). Then compute a 5% markup total two ways: a plain Python `for` loop over `big_amounts.values` (`total_loop`, timed as `loop_time`), and the vectorized `(big_amounts * 1.05).sum()` (`total_vectorized`, timed as `vector_time`). Confirm the totals match and compare the times.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
big_amounts = pd.Series(np.tile(df["loan_amount"].fillna(0).values, 2000))

start = time.time()
total_loop = 0
for val in big_amounts.values:
    total_loop += val * 1.05
loop_time = time.time() - start

start = time.time()
total_vectorized = (big_amounts * 1.05).sum()
vector_time = time.time() - start

print(round(total_loop, 2) == round(total_vectorized, 2), "loop:", loop_time, "vector:", vector_time)

**Q18.** `.apply()` looks vectorized (no explicit `for` keyword) but is actually still calling a Python function once per element under the hood — it's a middle ground, not a real vectorized operation. On `big_amounts`, compute the same 5% markup with `.apply(lambda x: x * 1.05)` (`apply_result`, timed as `apply_time`) versus the plain vectorized `big_amounts * 1.05` (`vectorized_result`, timed as `vector_time2`). Confirm they're equal and compare the times.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
start = time.time()
apply_result = big_amounts.apply(lambda x: x * 1.05)
apply_time = time.time() - start

start = time.time()
vectorized_result = big_amounts * 1.05
vector_time2 = time.time() - start

print(apply_result.equals(vectorized_result), "apply:", apply_time, "vector:", vector_time2)

**Q19.** `.iterrows()` is the classic pandas anti-pattern — it reconstructs a full Series object for every single row, which is expensive. On the original `df`, sum `loan_amount` by manually looping with `.iterrows()` (`manual_total`, timed as `iterrows_time`) versus just calling `.sum()` (`sum_total`, timed as `sum_time`). Confirm they match and compare — the gap should be dramatic even on this small dataset.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
start = time.time()
manual_total = 0
for idx, row in df.iterrows():
    manual_total += row["loan_amount"]
iterrows_time = time.time() - start

start = time.time()
sum_total = df["loan_amount"].sum()
sum_time = time.time() - start

print(round(manual_total, 2) == round(sum_total, 2), "iterrows:", iterrows_time, "sum:", sum_time)

**Q20.** Boolean masking (Phase 2) *is* vectorization — recap it here explicitly: build `high_value_mask = df["loan_amount"] > 20000` (a full-column comparison, no loop), then filter into `high_value_loans = df[high_value_mask]`. Print the shape.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
high_value_mask = df["loan_amount"] > 20000
high_value_loans = df[high_value_mask]
print(high_value_loans.shape)

**Q21.** String operations are vectorized too, through the `.str` accessor — `.str.upper()` runs the uppercase operation across the whole column at once instead of looping over Python strings. Uppercase `employment_status` into `upper_employment` and print the first 3 values.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
upper_employment = df["employment_status"].str.upper()
print(upper_employment.head(3).tolist())

## Topic 0E: The Index Object

Every DataFrame and Series has an Index, even if you never set one explicitly — you've been relying on it silently since Phase 1 for alignment, `.loc`, and merges.

**Q22.** Print the **type** of `df.index` (not its values — its type). By default, pandas gives you a `RangeIndex`: a memory-efficient stand-in for `0, 1, 2, ..., n-1` that doesn't actually store every integer.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
default_index_type = type(df.index).__name__
print(default_index_type)

**Q23.** Set `application_id` as the index using `.set_index("application_id")`, into `df_id_indexed`. Print the new index's type and its `.name` — once it's holding actual data instead of a simple range, it becomes a plain `Index` rather than a `RangeIndex`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df_id_indexed = df.set_index("application_id")
new_index_type = type(df_id_indexed.index).__name__
print(new_index_type, df_id_indexed.index.name)

**Q24.** From `df_id_indexed`, call `.reset_index(drop=True)` into `df_reset_dropped` (throws the old index away entirely, back to a plain `RangeIndex`), and `.reset_index()` with no arguments into `df_reset_kept` (moves the old index back into a regular column instead of discarding it). Confirm `"application_id"` is a column in the second but not the first.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
df_reset_dropped = df_id_indexed.reset_index(drop=True)
df_reset_kept = df_id_indexed.reset_index()
print("application_id" in df_reset_dropped.columns, "application_id" in df_reset_kept.columns)

**Q25.** This ties Topic 0A back together at the DataFrame level. Given `income_a` (index `A101, A102, A103`) and `income_b` (the **same three labels, but in a different order**: `A103, A101, A102`) below, compute `aligned_result = income_a + income_b`. Notice pandas lines them up by **label**, not by position — the row-order mismatch doesn't matter at all.

In [ ]:
income_a = pd.Series([50000, 60000, 70000], index=["A101", "A102", "A103"])
income_b = pd.Series([1000, 2000, 3000], index=["A103", "A101", "A102"])

# YOUR CODE HERE


**Solution**

In [ ]:
aligned_result = income_a + income_b
print(aligned_result)

**Q26.** Check whether `df_id_indexed`'s index has any duplicate labels using `.index.is_unique`, into `index_is_unique`. (A non-unique index is a common, quiet source of bugs — `.loc[some_id]` would return *multiple* rows instead of one if IDs repeated. Recall Phase 3 introduced exactly that scenario with re-submitted `application_id`s.)

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
index_is_unique = df_id_indexed.index.is_unique
print(index_is_unique)

## ✅ Checkpoint

**What you covered:**
- The Series: labeled 1D data (`.values` + `.index`), label vs. position access, and the critical **index-aligned arithmetic** behavior that makes pandas different from raw numpy
- The DataFrame: a dict of aligned Series, built from a dict-of-lists, list-of-dicts, explicit index, or directly from existing Series
- Methods vs. attributes: no-parens stored properties (`.shape`, `.columns`, `.dtypes`) vs. callable behavior (`.head()`, `.sum()`, `.sort_values()`), settable attributes, and `inplace=True` vs. reassignment
- Vectorization: loop vs. vectorized vs. `.apply()` timing, why `.iterrows()` is a red flag, and boolean masking / `.str` accessor as vectorization you'd already been using
- The Index: `RangeIndex` vs. a real `Index`, `.set_index()`/`.reset_index()` (with and without `drop=`), label-based alignment at the DataFrame level, and checking `.is_unique`

**Why it matters for the project:** this is the "why" behind everything you did in Phases 1-6 — why `.loc` and merges work the way they do, why `df = df.rename(...)` was necessary, and why every solution in this whole series has been vectorized instead of looped.

**What's next:** Phase 7, whenever you're ready — let me know the topics you want covered.